# Quantum Fourier Transform

The QFT maps a computational basis state |j⟩ to a superposition of all states with phases determined by the binary representation of j. Implemented on 3 qubits.

In [ ]:
import pennylane as qml
import numpy as np

## QFT circuit

Hadamard + controlled-RZ rotations, followed by SWAP to reverse qubit order.

In [ ]:
N = 3
dev = qml.device("default.qubit", wires=N)

def qft_rotations(wires):
    n = len(wires)
    for i in range(n):
        qml.Hadamard(wires=wires[i])
        for j in range(i + 1, n):
            k = j - i
            qml.ctrl(qml.RZ, control=wires[j])(np.pi / (2**k), wires=wires[i])

def qft_circuit():
    qft_rotations(list(range(N)))
    for i in range(N // 2):
        qml.SWAP(wires=[i, N - 1 - i])

@qml.qnode(dev)
def qft_matrix_check():
    qft_circuit()
    return qml.state()

state0 = qft_matrix_check()
expected = np.ones(2**N) / np.sqrt(2**N)
print("QFT|0\u27e9 amplitudes:")
print([f"{a:.4f}" for a in state0])
print(f"Equal superposition: {np.allclose(np.abs(state0), np.abs(expected))}")

## QFT on computational basis states

In [ ]:
@qml.qnode(dev)
def apply_qft(state):
    for i in range(N):
        if state & (1 << i):
            qml.PauliX(wires=i)
    qft_circuit()
    return qml.state()

for s in range(2**N):
    state = apply_qft(s)
    bits = format(s, f"0{N}b")
    print(f"\nQFT|{bits}\u27e9:")
    for i, amp in enumerate(state):
        if abs(amp) > 1e-10:
            ibits = format(i, f"0{N}b")
            print(f"  |{ibits}\u27e9: {amp:.4f}")